In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [ ]:
import sys
import subprocess

print("Python:", sys.executable)

# Install packages into the current notebook kernel's Python environment
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--user", "-U",
    "python-woc", "pandas", "matplotlib"
])

# Test the WoC import
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

print("\nSUCCESS: python-woc imported correctly!")
print("python-woc is ready to use.")

In [ ]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

# Create the WoC client
woc = WocMapsRemote(
    base_url="https://worldofcode.org/api/"
)

# Your 10 assigned projects
projects = [
    "comit-network/xmr-btc-swap",
    "systemPipeR/systemPipeShiny",
    "mmorise/World",
    "easystats/insight",
    "BMClab/BMC",
    "BRiAl/BRiAl",
    "JuliaRobotics/RigidBodyDynamics.jl",
    "Angel-ML/angel",
    "Samreay/ChainConsumer",
    "jasp-stats/jasp-desktop"
]

list_df_commits = []

for prj in projects:
    # Convert GitHub repository names to WoC V2412 project names
    prj = prj.lower().replace("/", "_", 2)

    print(f"Retrieving commits for {prj}...")

    commits = woc.get_values("p2c", prj)

    df = pd.DataFrame(commits, columns=["sha1"])
    df["project"] = prj

    list_df_commits.append(df)

# Combine all 10 projects
df = pd.concat(list_df_commits, ignore_index=True)

# Save retrieved commit list
df.to_csv("df_commits.csv", index=False)

# Show first record
df.head(1)

In [10]:
import time
from tqdm import tqdm
import pandas as pd

chunks = [
    df["sha1"].iloc[x:x+10]
    for x in range(0, len(df), 10)
]

commit_data = []

for i, chunk in enumerate(tqdm(chunks)):

    try:
        res, err = woc.get_values_many(
            "commit.tch",
            chunk.tolist()
        )

        if err:
            print(f"\nErrors on batch {i}: {err}")

        res = {k: v[0] for k, v in res.items()}

        for commit_sha, commit in res.items():
            commit_data.append({
                "commit": commit_sha,
                "tree": commit[0],
                "parent": list(commit[1]),
                "author": commit[2][0],
                "author_time": int(commit[2][1]),
                "author_tz": commit[2][2],
                "committer": commit[3][0],
                "committer_time": int(commit[3][1]),
                "committer_tz": commit[3][2],
                "message": commit[4],
            })

    except Exception as e:
        print(f"\nBatch {i} failed: {e}")
        break

    # Save every 1000 commits
    if (i + 1) % 100 == 0:
        pd.DataFrame(commit_data).to_csv(
            "commit_data_progress.csv",
            index=False
        )
        print(f"\nSaved {len(commit_data):,} commits")

    # Respect WoC rate limits
    time.sleep(1)

print(f"\nRetrieved {len(commit_data):,} commits")

  2%|▏         | 99/5012 [02:00<1:25:10,  1.04s/it]


Saved 1,000 commits


  4%|▍         | 199/5012 [03:47<1:23:17,  1.04s/it]


Saved 2,000 commits


  6%|▌         | 299/5012 [05:31<1:21:15,  1.03s/it]

...


Errors on batch 4995: {'fe2830a0c87e2e236b212632ec522733bffb7808': 'Key fe2830a0c87e2e236b212632ec522733bffb7808 not found in /da5_fast/All.sha1c/commit_126.tch'}


100%|█████████▉| 4999/5012 [1:27:00<00:13,  1.05s/it]


Saved 49,894 commits


100%|██████████| 5012/5012 [1:27:14<00:00,  1.04s/it]


Retrieved 50,009 commits


In [11]:
# What does the first record look like?

v = commit_data[0]

# commit tree, [parent{s}], [author, unixtime, tz],
# [committer, unixtime, tz], commit message
print(v)

{'commit': '00061307563ba77ae3927cdbad33de13c2084327', 'tree': '01ec734cc33c4c2bbafc3eaae9cfb6581f8a8519', 'parent': ['7c18ae23a8a8826fa41deedfd71a5b7d60ea31dd', '7f74fd0190dbb506a4d4bc7dc456885e2b9e0060'], 'author': 'bors[bot] <26634292+bors[bot]@users.noreply.github.com>', 'author_time': 1620258526, 'author_tz': '+0000', 'committer': 'GitHub <noreply@github.com>', 'committer_time': 1620258526, 'committer_tz': '+0000', 'message': 'Merge #473\n\n473: Bump rust_decimal from 1.12.4 to 1.13.0 r=thomaseizinger a=dependabot[bot]\n\nBumps [rust_decimal](https://github.com/paupino/rust-decimal) from 1.12.4 to 1.13.0.\n<details>\n<summary>Commits</summary>\n<ul>\n<li>See full diff in <a href="https://github.com/paupino/rust-decimal/commits/1.13.0">compare view</a></li>\n</ul>\n</details>\n<br />\n\n\n[![Dependabot compatibility score](https://dependabot-badges.githubapp.com/badges/compatibility_score?dependency-name=rust_decimal&package-manager=cargo&previous-version=1.12.4&new-version=1.13.0)

In [12]:
# Convert the retrieved commit data into a dataframe

df_commit_data = pd.DataFrame(commit_data)

# Add the project associated with each commit
df_commit_data = df_commit_data.merge(
    df,
    left_on="commit",
    right_on="sha1"
)

# Save the retrieved commit data
df_commit_data.to_csv(
    "df_commit_data.csv",
    index=False
)

df_commit_data.head(2)


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00061307563ba77ae3927cdbad33de13c2084327,01ec734cc33c4c2bbafc3eaae9cfb6581f8a8519,"[7c18ae23a8a8826fa41deedfd71a5b7d60ea31dd, 7f7...",bors[bot] <26634292+bors[bot]@users.noreply.gi...,1620258526,+0000,GitHub <noreply@github.com>,1620258526,+0000,Merge #473\n\n473: Bump rust_decimal from 1.12...,00061307563ba77ae3927cdbad33de13c2084327,comit-network_xmr-btc-swap
1,002e7b38c39bbb76fccde34b0f1838c316eaa8f1,e8016cc91d6132472f611f7e8cb3930ee387f46d,[00061307563ba77ae3927cdbad33de13c2084327],Philipp Hoenisch <philipp@hoenisch.at>,1619585554,+1000,Philipp Hoenisch <philipp@hoenisch.at>,1620347081,+1000,Define TX_FEE within test files only.\n,002e7b38c39bbb76fccde34b0f1838c316eaa8f1,comit-network_xmr-btc-swap


In [13]:
# Create the final project summary CSV

df_final = df_commit_data[
    ["project", "commit", "author", "author_time", "message"]
].copy()

df_final.columns = [
    "project_wocid",
    "commit_sha1",
    "author",
    "time",
    "commit message"
]

df_final.to_csv(
    "jmurph91_project_summary.csv",
    index=False,
    sep=";"
)

print(f"Saved {len(df_final):,} commits")
df_final.head(1)

Saved 50,009 commits


,project_wocid,commit_sha1,author,time,commit message
0,comit-network_xmr-btc-swap,00061307563ba77ae3927cdbad33de13c2084327,bors[bot] <26634292+bors[bot]@users.noreply.gi...,1620258526,Merge #473\n\n473: Bump rust_decimal from 1.12...


In [14]:
# Verify the final CSV has the required format

print("Columns:")
print(list(df_final.columns))

print(f"\nNumber of rows: {len(df_final):,}")

print("\nFirst row:")
display(df_final.head(1))

Columns:
['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']

Number of rows: 50,009

First row:


,project_wocid,commit_sha1,author,time,commit message
0,comit-network_xmr-btc-swap,00061307563ba77ae3927cdbad33de13c2084327,bors[bot] <26634292+bors[bot]@users.noreply.gi...,1620258526,Merge #473\n\n473: Bump rust_decimal from 1.12...


In [15]:
# Calculate WoC statistics for each project

woc_summary = (
    df_final.groupby("project_wocid")
    .agg(
        commits=("commit_sha1", "count"),
        authors=("author", "nunique"),
        min_time=("time", "min"),
        max_time=("time", "max")
    )
    .reset_index()
)

woc_summary

,project_wocid,commits,authors,min_time,max_time
0,angel-ml_angel,3418,147,1493817527,1760616761
1,bmclab_bmc,2083,30,1383617973,1762230919
2,brial_brial,2999,34,1141893962,1759360722
3,comit-network_xmr-btc-swap,4980,68,1601272508,1755593438
4,easystats_insight,9905,86,1548793879,1776168259
5,jasp-stats_jasp-desktop,22023,242,1370113512,1777093856
6,juliarobotics_rigidbodydynamics.jl,2984,23,1460686712,1740965887
7,mmorise_world,348,46,1447574109,1740126621
8,samreay_chainconsumer,634,18,1468796482,1775513079
9,systempiper_systempipeshiny,635,20,1582064135,1747714763


In [16]:
# Add readable versions of the minimum and maximum commit times

woc_summary["min_time_date"] = pd.to_datetime(
    woc_summary["min_time"],
    unit="s"
)

woc_summary["max_time_date"] = pd.to_datetime(
    woc_summary["max_time"],
    unit="s"
)

woc_summary

,project_wocid,commits,authors,min_time,max_time,min_time_date,max_time_date
0,angel-ml_angel,3418,147,1493817527,1760616761,2017-05-03 13:18:47,2025-10-16 12:12:41
1,bmclab_bmc,2083,30,1383617973,1762230919,2013-11-05 02:19:33,2025-11-04 04:35:19
2,brial_brial,2999,34,1141893962,1759360722,2006-03-09 08:46:02,2025-10-01 23:18:42
3,comit-network_xmr-btc-swap,4980,68,1601272508,1755593438,2020-09-28 05:55:08,2025-08-19 08:50:38
4,easystats_insight,9905,86,1548793879,1776168259,2019-01-29 20:31:19,2026-04-14 12:04:19
5,jasp-stats_jasp-desktop,22023,242,1370113512,1777093856,2013-06-01 19:05:12,2026-04-25 05:10:56
6,juliarobotics_rigidbodydynamics.jl,2984,23,1460686712,1740965887,2016-04-15 02:18:32,2025-03-03 01:38:07
7,mmorise_world,348,46,1447574109,1740126621,2015-11-15 07:55:09,2025-02-21 08:30:21
8,samreay_chainconsumer,634,18,1468796482,1775513079,2016-07-17 23:01:22,2026-04-06 22:04:39
9,systempiper_systempipeshiny,635,20,1582064135,1747714763,2020-02-18 22:15:35,2025-05-20 04:19:23


In [17]:
# Final project summary
# GitHub information will be added to this table.

final_summary = woc_summary.copy()

final_summary

,project_wocid,commits,authors,min_time,max_time,min_time_date,max_time_date
0,angel-ml_angel,3418,147,1493817527,1760616761,2017-05-03 13:18:47,2025-10-16 12:12:41
1,bmclab_bmc,2083,30,1383617973,1762230919,2013-11-05 02:19:33,2025-11-04 04:35:19
2,brial_brial,2999,34,1141893962,1759360722,2006-03-09 08:46:02,2025-10-01 23:18:42
3,comit-network_xmr-btc-swap,4980,68,1601272508,1755593438,2020-09-28 05:55:08,2025-08-19 08:50:38
4,easystats_insight,9905,86,1548793879,1776168259,2019-01-29 20:31:19,2026-04-14 12:04:19
5,jasp-stats_jasp-desktop,22023,242,1370113512,1777093856,2013-06-01 19:05:12,2026-04-25 05:10:56
6,juliarobotics_rigidbodydynamics.jl,2984,23,1460686712,1740965887,2016-04-15 02:18:32,2025-03-03 01:38:07
7,mmorise_world,348,46,1447574109,1740126621,2015-11-15 07:55:09,2025-02-21 08:30:21
8,samreay_chainconsumer,634,18,1468796482,1775513079,2016-07-17 23:01:22,2026-04-06 22:04:39
9,systempiper_systempipeshiny,635,20,1582064135,1747714763,2020-02-18 22:15:35,2025-05-20 04:19:23


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## GitHub Repository Information

The following information was collected from the GitHub repositories for each of the ten assigned projects. The table reports the number of stars, number of forks, and the date of the most recent commit.

| Project | GitHub Repository | Stars | Forks | Last Commit Date |
|---|---|---:|---:|---|
| comit-network/xmr-btc-swap | https://github.com/comit-network/xmr-btc-swap | 712 | 182 | Jul 22, 2025 |
| systemPipeR/systemPipeShiny | https://github.com/systemPipeR/systemPipeShiny | 36 | 6 | May 20, 2025 |
| mmorise/World | https://github.com/mmorise/World | 1.3k | 264 | Feb 21, 2025 |
| easystats/insight | https://github.com/easystats/insight | 442 | 47 | Sep 16, 2026 |
| BMClab/BMC | https://github.com/BMClab/BMC | 467 | 264 | Sep 13, 2026 |
| BRiAl/BRiAl | https://github.com/BRiAl/BRiAl | 24 | 20 | Sep 1, 2026 |
| JuliaRobotics/RigidBodyDynamics.jl | https://github.com/JuliaRobotics/RigidBodyDynamics.jl | 310 | 53 | Nov 7, 2024 |
| Angel-ML/angel | https://github.com/Angel-ML/angel | 6.8k | 1.6k | Oct 13, 2025 |
| Samreay/ChainConsumer | https://github.com/Samreay/ChainConsumer | 91 | 21 | Apr 6, 2026 |
| jasp-stats/jasp-desktop | https://github.com/jasp-stats/jasp-desktop | 1.0k | 231 | Sep 25, 2026 |